# Distilling a Qwen3-32B teacher into Qwen3-14B on τ-bench retail

**One experiment, all numbers recomputed from the files in this repo** (`results/distill/*.json`, `dataset/distill/**`).

1. A **Qwen3-32B-AWQ teacher** plays the 114 hard retail *train* tasks (the ones the 14B solves rarely or never, plus mixed ones); only its **passing** conversations are kept (180 conversations).
2. The 14B (QLoRA r=16, lr 5e-5, 1 epoch = 384 steps) is fine-tuned on those conversations, loss on assistant turns only.
3. Both the new model and the **starting adapter (control, re-run on the same pod)** are tested on the 115-task retail test split, **4 trials per task**, temperature 0.7, GPT-4o as the simulated customer.

**Honest summary of the result:** the new model scores +3.9 points pass^1 over the same-pod control (45.7% vs 41.7%), which is *not* statistically significant (paired permutation p = 0.15). Against the earlier baseline measurement (37.8%, other pod, 2 trials) the gap looks nominally significant (+8.7 points on the matched first two trials, p = 0.03), but the identical starting adapter re-run on the new pod scored +2.2 points higher than that earlier test (not significant by itself), so part of that gap is pod-to-pod drift. All three comparisons are shown below.

In [1]:
import json, math, random
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats

pio.renderers.default = "notebook_connected"
pio.templates.default = "plotly_white"
ROOT = Path("..") if Path("../results/distill").exists() else Path(".")
RUN = json.load(open(ROOT / "results/distill/distill_run.json"))
TEACHER = json.load(open(ROOT / "results/distill/teacher_run.json"))
CONV = ROOT / "dataset/distill/retail_eval_conversations"
COMBO = {t["task_index"] for t in json.load(open(ROOT / "dataset/distill/combo_tasks_test_64.json"))}

def load_results(name):
    rows = [json.loads(l) for l in open(CONV / name)]
    by = defaultdict(dict)
    for r in rows:
        by[r["task_index"]][r["trial"]] = 1 if r["reward"] >= 1 else 0
    return rows, {t: [v[k] for k in sorted(v)] for t, v in by.items()}

NEW_ROWS, NEW = load_results("retail_new_model_4trials_results.jsonl")
CTL_ROWS, CTL = load_results("retail_starting_adapter_rerun_4trials_results.jsonl")
_old = defaultdict(list)
for r in json.load(open(ROOT / "results/passk_run.json"))["models"][1]["rows"]:   # starting adapter, earlier test, 2 trials, other pod
    _old[r["task_index"]].append((r["trial"], 1 if r["reward"] >= 1 else 0))
EARLIER = {t: [x for _, x in sorted(v)] for t, v in _old.items()}
TASKS = sorted(NEW)
C_NEW, C_CTL, C_OLD, C_TEACH = "#1E7A5C", "#E0883B", "#2F5D8A", "#7A4E9C"
print(f"new model: {len(NEW_ROWS)} rollouts, control: {len(CTL_ROWS)} rollouts, tasks: {len(TASKS)}, combo tasks in test: {len(COMBO)}")

new model: 460 rollouts, control: 460 rollouts, tasks: 115, combo tasks in test: 64


In [2]:
def passk(v, k):
    n = len(v)
    return math.comb(sum(v), k) / math.comb(n, k) if n >= k else None

def per_task(d, k):
    return {t: passk(v, k) for t, v in d.items() if passk(v, k) is not None}

def mean_se(x):
    x = list(x)
    n = len(x)
    if not n:
        return None, None
    m = sum(x) / n
    return m, (math.sqrt(sum((a - m) ** 2 for a in x) / (n - 1) / n) if n > 1 else None)

def summary(d):
    out = {}
    for k in (1, 2, 3, 4):
        m, se = mean_se(per_task(d, k).values())
        out[k] = (m, se)
    return out

SUM = {"new model": summary(NEW), "control (same pod)": summary(CTL), "earlier test (other pod, 2 trials)": summary(EARLIER)}
pd.DataFrame({n: {f"pass^{k}": (f"{m:.1%} +/- {se:.1%}" if m is not None else "-") for k, (m, se) in s.items()} for n, s in SUM.items()})

,new model,control (same pod),"earlier test (other pod, 2 trials)"
pass^1,45.7% +/- 3.4%,41.7% +/- 3.4%,37.8% +/- 3.7%
pass^2,29.9% +/- 3.4%,27.1% +/- 3.4%,21.7% +/- 3.9%
pass^3,22.0% +/- 3.4%,20.2% +/- 3.4%,-
pass^4,16.5% +/- 3.5%,16.5% +/- 3.5%,-


## 1. Teacher: which hard tasks did Qwen3-32B solve?

In [3]:
S = TEACHER["summary"]
print(f"{S['model']}: {S['n_tasks']} tasks, {S['rollouts_done']} rollouts, {S['rollouts_passed']} passed, "
      f"{S['tasks_solved']} tasks solved ({S['tasks_solved']/S['n_tasks']:.0%}); sampling of a task stopped after {S['stop_after_passes']} passes")
pt = pd.DataFrame(TEACHER["per_task"])
g = pt.groupby("group_14b").agg(tasks=("task_index", "count"), solved=("solved", "sum")).reindex(["never", "mixed", "always"]).dropna()
g["unsolved"] = g["tasks"] - g["solved"]
fig = make_subplots(rows=1, cols=3, subplot_titles=("Tasks the teacher solved, by how the 14B did before", "Passes per task (of the rollouts played)", "Cumulative teacher passes over time"),
                    column_widths=[0.33, 0.33, 0.34])
fig.add_bar(x=g.index, y=g["solved"], name="solved by teacher", marker_color=C_NEW, row=1, col=1)
fig.add_bar(x=g.index, y=g["unsolved"], name="not solved", marker_color="#B8B8B8", row=1, col=1)
cnt = pt["passes"].value_counts().sort_index()
fig.add_bar(x=cnt.index.astype(str), y=cnt.values, marker_color=C_TEACH, showlegend=False, row=1, col=2)
rows = sorted((r for r in TEACHER["_rows"] if not r.get("skipped")), key=lambda r: r["finished_at"])
t0 = rows[0]["started_run_at"]
cum = np.cumsum([1 if r["reward"] >= 1 else 0 for r in rows])
fig.add_scatter(x=[(r["finished_at"] - t0) / 60 for r in rows], y=cum, mode="lines", line=dict(color=C_TEACH), showlegend=False, row=1, col=3)
fig.update_xaxes(title_text="14B group", row=1, col=1); fig.update_xaxes(title_text="passes", row=1, col=2); fig.update_xaxes(title_text="minutes", row=1, col=3)
fig.update_layout(barmode="stack", height=420, width=1250, legend=dict(orientation="h", y=-0.25))
fig.show()

Qwen3-32B-AWQ: 114 tasks, 419 rollouts, 199 passed, 86 tasks solved (75%); sampling of a task stopped after 2 passes


## 2. SFT on the 180 passing conversations

In [4]:
T = RUN["training"]
step = T["step_index"]
fig = make_subplots(rows=1, cols=3, subplot_titles=("Loss (raw and 20-step average)", "Gradient norm", "Learning rate"))
loss = np.array(T["loss"], dtype=float)
fig.add_scatter(x=step, y=loss, mode="lines", line=dict(color="#BBBBBB", width=1), name="loss", row=1, col=1)
sm = np.convolve(loss, np.ones(20) / 20, mode="valid")
fig.add_scatter(x=step[19:], y=sm, mode="lines", line=dict(color=C_NEW, width=3), name="20-step average", row=1, col=1)
fig.add_scatter(x=step, y=T["grad_norm"], mode="lines", line=dict(color=C_CTL), showlegend=False, row=1, col=2)
fig.add_scatter(x=step, y=T["lr"], mode="lines", line=dict(color=C_OLD), showlegend=False, row=1, col=3)
for ck in (96, 192, 288):
    fig.add_vline(x=ck, line_dash="dot", line_color="#999", row=1, col=1)
fig.update_xaxes(title_text="optimizer step")
fig.update_layout(height=400, width=1250, legend=dict(orientation="h", y=-0.25))
fig.show()
print(f"{T['steps_total']} steps, {T['elapsed_min']/60:.1f} h, loss first 10 steps {T['loss_first10']:.2f} -> last 10 steps {T['loss_last10']:.2f}")

384 steps, 3.2 h, loss first 10 steps 0.53 -> last 10 steps 0.35


## 3. Retail test: new model vs the control
Both ran on the same pod with identical settings (vLLM 0.11.0, adapters loaded side by side). pass^k per task = C(c,k)/C(n,k) for c passes in n = 4 trials; error bars are ±1 standard error over tasks.

In [5]:
fig = make_subplots(rows=1, cols=3, column_widths=[0.38, 0.32, 0.30],
                    subplot_titles=("Running pass^1 in completion order", "pass^k, all 115 tasks (±1 s.e.)", "pass^1 per trial"))
def running(rows):
    rs = sorted(rows, key=lambda r: r["finished_at"])
    return np.cumsum([1 if r["reward"] >= 1 else 0 for r in rs]) / np.arange(1, len(rs) + 1)
for name, color, rows in (("new model", C_NEW, NEW_ROWS), ("control, same pod", C_CTL, CTL_ROWS)):
    fig.add_scatter(y=running(rows), mode="lines", line=dict(color=color), name=name, row=1, col=1)
old = RUN["baseline"].get("running_pass1")
if old:
    fig.add_scatter(y=old, mode="lines", line=dict(color=C_OLD, dash="dot"), name="earlier test (2 trials)", row=1, col=1)
for name, color in (("earlier test (other pod, 2 trials)", C_OLD), ("control (same pod)", C_CTL), ("new model", C_NEW)):
    s = SUM[name]
    xs = [f"pass^{k}" for k in (1, 2, 3, 4) if s[k][0] is not None]
    ys = [s[k][0] for k in (1, 2, 3, 4) if s[k][0] is not None]
    es = [s[k][1] for k in (1, 2, 3, 4) if s[k][0] is not None]
    fig.add_bar(x=xs, y=ys, error_y=dict(type="data", array=es), name=name, marker_color=color, text=[f"{y:.0%}" for y in ys], textposition="outside", row=1, col=2)
for name, color, rows in (("new model", C_NEW, NEW_ROWS), ("control, same pod", C_CTL, CTL_ROWS)):
    tr = defaultdict(list)
    for r in rows:
        tr[r["trial"]].append(1 if r["reward"] >= 1 else 0)
    fig.add_scatter(x=[f"trial {k+1}" for k in sorted(tr)], y=[sum(tr[k]) / len(tr[k]) for k in sorted(tr)], mode="lines+markers", line=dict(color=color), showlegend=False, row=1, col=3)
fig.update_yaxes(range=[0, 0.8])
fig.update_xaxes(title_text="rollouts finished", row=1, col=1)
fig.update_layout(height=450, width=1350, barmode="group", legend=dict(orientation="h", y=-0.2))
fig.show()

In [6]:
fig = make_subplots(rows=1, cols=4, subplot_titles=[f"pass^{k} by task type" for k in (1, 2, 3, 4)], shared_yaxes=True)
for ci, k in enumerate((1, 2, 3, 4), 1):
    for name, color, d in (("control, same pod", C_CTL, CTL), ("new model", C_NEW, NEW)):
        ys, es = [], []
        for keep in (lambda t: t in COMBO, lambda t: t not in COMBO):
            m, se = mean_se(v for t, v in per_task({t: x for t, x in d.items() if keep(t)}, k).items())
            ys.append(m); es.append(se)
        fig.add_bar(x=[f"combo tasks ({len(COMBO)})", f"other tasks ({len(TASKS) - len(COMBO)})"], y=ys, error_y=dict(type="data", array=es),
                    marker_color=color, name=name, showlegend=(ci == 1), text=[f"{y:.0%}" for y in ys], textposition="outside", row=1, col=ci)
fig.update_yaxes(range=[0, 0.8])
fig.update_layout(height=420, width=1350, barmode="group", legend=dict(orientation="h", y=-0.2))
fig.show()

## 4. Is the difference real? Paired by task

In [7]:
rng = np.random.default_rng(0)
def paired(a, b, k=1):
    xs = np.array([np.mean(a[t]) - np.mean(b[t]) if k == 1 else passk(a[t], k) - passk(b[t], k) for t in TASKS if t in a and t in b
                   and (k == 1 or (passk(a[t], k) is not None and passk(b[t], k) is not None))])
    n = len(xs); m = xs.mean(); se = xs.std(ddof=1) / math.sqrt(n)
    boots = np.array([rng.choice(xs, n).mean() for _ in range(20000)])
    flips = np.array([(xs * rng.choice([-1, 1], n)).mean() for _ in range(20000)])
    p = float((np.abs(flips) >= abs(m)).mean())
    return dict(n=n, mean=m, se=se, lo=float(np.percentile(boots, 2.5)), hi=float(np.percentile(boots, 97.5)), p=p, xs=xs, boots=boots)

vs_ctl = paired(NEW, CTL, 1)
vs_old = paired({t: v[:2] for t, v in NEW.items()}, EARLIER, 1)   # new model's first 2 trials vs the earlier 2-trial test
vs_rerun = paired(CTL, EARLIER, 1) if False else paired({t: v[:2] for t, v in CTL.items()}, EARLIER, 1)
vs_ctl2 = paired(NEW, CTL, 2)
tbl = pd.DataFrame([
    ["new model vs control (same pod), pass^1, 4 trials", vs_ctl], ["new model vs control (same pod), pass^2", vs_ctl2],
    ["new model (first 2 trials) vs EARLIER test, pass^1", vs_old], ["CONTROL (first 2 trials) vs EARLIER test, pass^1", vs_rerun]],
    columns=["comparison", "r"])
for c in ("mean", "se", "lo", "hi", "p"):
    tbl[c] = tbl["r"].map(lambda r: r[c])
tbl["tasks"] = tbl["r"].map(lambda r: r["n"])
tbl = tbl.drop(columns="r")
tbl.style.format({"mean": "{:+.3f}", "se": "{:.3f}", "lo": "{:+.3f}", "hi": "{:+.3f}", "p": "{:.3f}"}).hide(axis="index")

comparison,mean,se,lo,hi,p,tasks
"new model vs control (same pod), pass^1, 4 trials",+0.039,0.026,-0.011,+0.089,0.145,115
"new model vs control (same pod), pass^2",+0.028,0.029,-0.028,+0.084,0.341,115
"new model (first 2 trials) vs EARLIER test, pass^1",+0.087,0.038,+0.013,+0.161,0.034,115
"CONTROL (first 2 trials) vs EARLIER test, pass^1",+0.022,0.038,-0.052,+0.096,0.647,115


In [8]:
# sign test on pass^2 (the same test the notebook-in-progress reported)
pa, pb = per_task(NEW, 2), per_task(CTL, 2)
wins = sum(pa[t] > pb[t] for t in TASKS); losses = sum(pa[t] < pb[t] for t in TASKS)
sign_p = stats.binomtest(wins, wins + losses, 0.5).pvalue
print(f"pass^2 sign test: new higher on {wins} tasks, control higher on {losses}, {len(TASKS)-wins-losses} ties, p = {sign_p:.2f}")

fig = make_subplots(rows=1, cols=3, column_widths=[0.34, 0.33, 0.33],
                    subplot_titles=("Per-task pass rate: new model vs control", "Bootstrap of the mean difference (new - control)", "Effect with 95% intervals"))
rs = np.random.default_rng(1)
xj = np.array([np.mean(CTL[t]) for t in TASKS]) + rs.uniform(-0.03, 0.03, len(TASKS))
yj = np.array([np.mean(NEW[t]) for t in TASKS]) + rs.uniform(-0.03, 0.03, len(TASKS))
fig.add_scatter(x=xj, y=yj, mode="markers", marker=dict(size=7, color=[C_TEACH if t in COMBO else C_NEW for t in TASKS], opacity=0.7),
                text=[f"task {t}{' (combo)' if t in COMBO else ''}: control {np.mean(CTL[t]):.2f}, new {np.mean(NEW[t]):.2f}" for t in TASKS], hoverinfo="text", showlegend=False, row=1, col=1)
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color="#999", dash="dot"), showlegend=False, row=1, col=1)
fig.add_histogram(x=vs_ctl["boots"], nbinsx=60, marker_color=C_NEW, showlegend=False, row=1, col=2)
fig.add_vline(x=0, line_color="black", row=1, col=2)
fig.add_vline(x=vs_ctl["mean"], line_color=C_CTL, line_dash="dash", row=1, col=2)
labels = ["new vs control (4 trials)", "new vs earlier (2 trials)", "control vs earlier (2 trials)"]
for lab, r, color in zip(labels, (vs_ctl, vs_old, vs_rerun), (C_NEW, C_OLD, C_CTL)):
    fig.add_scatter(x=[r["mean"]], y=[lab], mode="markers", marker=dict(color=color, size=11), error_x=dict(type="data", symmetric=False, array=[r["hi"] - r["mean"]], arrayminus=[r["mean"] - r["lo"]]),
                    showlegend=False, text=[f"p = {r['p']:.3f}"], hoverinfo="text+x", row=1, col=3)
fig.add_vline(x=0, line_color="black", row=1, col=3)
fig.update_xaxes(title_text="control pass rate", row=1, col=1); fig.update_yaxes(title_text="new model pass rate", row=1, col=1)
fig.update_xaxes(title_text="mean pass^1 difference", row=1, col=2); fig.update_xaxes(title_text="pass^1 difference", row=1, col=3)
fig.update_layout(height=470, width=1350)
fig.show()

pass^2 sign test: new higher on 31 tasks, control higher on 23, 61 ties, p = 0.34


**How to read the right-hand panel.** The new model's gain over the *same-pod control* (4 trials) has an interval that includes zero. Its gain over the *earlier test* uses only the first 2 trials of each model so the trial counts match: it is larger (+8.7 points, p = 0.03), but the control re-run on the new pod already sits +2.2 points above the earlier test, so some of that is pod drift. The same-pod control is the comparison to trust. Note also that the new model's first two trials beat the control by +6.5 points, and trials 3 and 4 by only +1.3: the early lead partly regressed with more sampling.

## 5. Where does the difference come from?

In [9]:
def delta_by(group_of):
    out = defaultdict(list)
    for t in TASKS:
        out[group_of(t)].append(np.mean(NEW[t]) - np.mean(CTL[t]))
    return out
d = delta_by(lambda t: "combo tasks (trained on this kind)" if t in COMBO else "other tasks")
fig = make_subplots(rows=1, cols=3, column_widths=[0.3, 0.35, 0.35],
                    subplot_titles=("Mean pass^1 difference by task type", "Tasks gained / lost (new vs control)", "Conversation length (messages) per rollout"))
fig.add_bar(x=list(d), y=[np.mean(v) for v in d.values()], error_y=dict(type="data", array=[np.std(v, ddof=1) / math.sqrt(len(v)) for v in d.values()]),
            marker_color=[C_TEACH, C_NEW], showlegend=False, row=1, col=1)
delta = {t: np.mean(NEW[t]) - np.mean(CTL[t]) for t in TASKS}
bins = Counter(round(v * 4) / 4 for v in delta.values())
fig.add_bar(x=sorted(bins), y=[bins[k] for k in sorted(bins)], marker_color=[C_NEW if k > 0 else C_CTL if k < 0 else "#B8B8B8" for k in sorted(bins)], showlegend=False, row=1, col=2)
for name, color, rows in (("new model", C_NEW, NEW_ROWS), ("control", C_CTL, CTL_ROWS)):
    fig.add_histogram(x=[r["n_messages"] for r in rows], name=name, marker_color=color, opacity=0.6, nbinsx=30, row=1, col=3)
fig.update_xaxes(title_text="new - control pass rate (task level, trials averaged)", row=1, col=2); fig.update_yaxes(title_text="tasks", row=1, col=2)
fig.update_xaxes(title_text="messages in the conversation", row=1, col=3)
fig.update_layout(height=430, width=1350, barmode="overlay", legend=dict(orientation="h", y=-0.2))
fig.show()
print({k: f"{np.mean(v):+.3f} over {len(v)} tasks" for k, v in d.items()})

{'other tasks': '+0.074 over 51 tasks', 'combo tasks (trained on this kind)': '+0.012 over 64 tasks'}


## 6. Verdict
* The teacher solved 86 of 114 hard tasks (75%); its 180 passing conversations were the only training data.
* Against the **same-pod control** the 14B gained about **+4 points pass^1** (95% interval roughly -1 to +9, p ≈ 0.15), identical pass^4, same consistency: a plausible but unproven small effect.
* The apparent gain is concentrated in **non-combo** tasks even though training used only combo-task conversations, which hints that conversation handling transfers more than task-specific skill, but at this sample size that is a hypothesis, not a finding.
* The airline test was started and stopped after 10 rollouts: no airline comparison exists.

Raw conversations: `dataset/distill/retail_eval_conversations/`.